# Amazon Sales — Revenue & Customer Behavior Analysis

**Dataset:** [Amazon Sales Dataset](https://www.kaggle.com/datasets/aliiihussain/amazon-sales-dataset) by Ali Hussain (Kaggle)  
**Scope:** ~50,000 orders across 6 product categories, 4 global regions, 2022–2023  
**Goal:** Identify revenue drivers, pricing patterns, discount impact, and customer segments to support business decisions.

---

### Analysis outline
1. Setup & data loading  
2. Data quality check  
3. Descriptive statistics  
4. Business queries (revenue, trends, regions, discounts, ratings)  
5. Correlation analysis  
6. K-Means clustering: product behavior segments  
7. K-Means clustering: customer satisfaction segments  
8. Key findings & recommendations

## 1. Setup & data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
import warnings
import os

warnings.filterwarnings('ignore')

# clean chart style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_d')
plt.rcParams.update({'figure.dpi': 130, 'axes.titlesize': 13, 'axes.labelsize': 11})

# create output folder for saved images (used in README)
os.makedirs('images', exist_ok=True)

print('Libraries loaded successfully.')

In [ ]:
# ---- load dataset ----
# place the CSV inside a /data folder next to this notebook
df = pd.read_csv('data/amazon_sales.csv')

# parse dates so monthly aggregations work correctly
df['order_date'] = pd.to_datetime(df['order_date'])
df['month']      = df['order_date'].dt.to_period('M')
df['year_month'] = df['order_date'].dt.strftime('%m-%Y')

print(f'Rows: {df.shape[0]:,}   Columns: {df.shape[1]}')
df.head()

## 2. Data quality check

Before drawing any conclusions, verify the dataset is clean: correct data types, no unexpected nulls, no obvious outliers.

In [ ]:
print('--- Data types ---')
print(df.dtypes)

print('\n--- Missing values per column ---')
nulls = df.isnull().sum()
print(nulls[nulls > 0] if nulls.sum() > 0 else 'No missing values found.')

print('\n--- Duplicate rows ---')
print(f'{df.duplicated().sum()} duplicate rows')

print('\n--- Unique values in categorical columns ---')
for col in ['product_category', 'customer_region', 'payment_method']:
    print(f'  {col}: {sorted(df[col].unique())}')

## 3. Descriptive statistics

In [ ]:
# summary stats for all numeric columns
df.describe().round(2)

In [ ]:
# record counts per categorical attribute
for col in ['product_category', 'customer_region', 'payment_method']:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

## 4. Business queries

Each query answers a specific business question. After every chart, a brief **insight** statement summarizes the actionable finding.

### Query 1. Total revenue by product category
*Which categories drive the most revenue?*

In [ ]:
rev_cat = (
    df.groupby('product_category')['total_revenue']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
rev_cat['pct'] = (rev_cat['total_revenue'] / rev_cat['total_revenue'].sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(rev_cat['product_category'], rev_cat['total_revenue'],
              color=sns.color_palette('Blues_d', len(rev_cat)))
ax.set_title('Total Revenue by Product Category')
ax.set_xlabel('')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
for bar, pct in zip(bars, rev_cat['pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20000,
            f'{pct}%', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('images/01_revenue_by_category.png', dpi=150)
plt.show()

print(rev_cat.to_string(index=False))
print('\nInsight: Revenue is evenly distributed across all 6 categories (max gap: $143K).'
      ' This signals a balanced portfolio with no single-category dependency risk.')

### Query 2. Monthly revenue trend
*Is there seasonality? When are the revenue peaks?*

In [ ]:
monthly_rev = (
    df.groupby('month')['total_revenue']
    .sum()
    .reset_index()
)
monthly_rev['month_str'] = monthly_rev['month'].astype(str)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(len(monthly_rev)), monthly_rev['total_revenue'],
        color='#185FA5', marker='o', linewidth=2, markersize=4)
ax.set_title('Monthly Revenue Trend (Jan 2022 – Dec 2023)')
ax.set_xticks(range(len(monthly_rev)))
ax.set_xticklabels(monthly_rev['month_str'], rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.2f}M'))
ax.axhline(monthly_rev['total_revenue'].mean(), color='gray', linestyle='--',
           linewidth=1, label=f"Avg: ${monthly_rev['total_revenue'].mean()/1e6:.2f}M")
ax.legend()
plt.tight_layout()
plt.savefig('images/02_monthly_revenue.png', dpi=150)
plt.show()

print('Insight: January shows a post-holiday spike. February dips ~13%. '
      'Revenue stabilizes around $1.37M/month from March onward.')

### Query 3. Units sold by region
*Which regions have the highest sales volume?*

In [ ]:
units_region = (
    df.groupby('customer_region')['quantity_sold']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# bar chart
ax1.bar(units_region['customer_region'], units_region['quantity_sold'],
        color=sns.color_palette('Blues_d', len(units_region)))
ax1.set_title('Units Sold by Region')
ax1.set_xlabel('')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e3:.0f}K'))

# pie chart
ax2.pie(units_region['quantity_sold'], labels=units_region['customer_region'],
        autopct='%1.1f%%', startangle=90,
        colors=sns.color_palette('Blues_d', len(units_region)))
ax2.set_title('Regional Share of Units Sold')

plt.tight_layout()
plt.savefig('images/03_units_by_region.png', dpi=150)
plt.show()

print(units_region.to_string(index=False))
print('\nInsight: Regional distribution is nearly uniform (~25% each). '
      'No single market dominates, indicating a consistent global strategy.')

### Query 4. Payment method usage
*What are customer payment preferences?*

In [ ]:
pay_counts = df['payment_method'].value_counts().reset_index()
pay_counts.columns = ['payment_method', 'transactions']

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(pay_counts['payment_method'], pay_counts['transactions'],
        color=sns.color_palette('Blues_d', len(pay_counts)))
ax.set_title('Transactions by Payment Method')
ax.set_xlabel('Number of Transactions')
for i, v in enumerate(pay_counts['transactions']):
    ax.text(v + 20, i, f'{v:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('images/04_payment_methods.png', dpi=150)
plt.show()

print(pay_counts.to_string(index=False))
print('\nInsight: All 5 methods are used almost equally (~20% each). '
      'Wallet and UPI lead slightly, reflecting strong digital payment adoption.')

### Query 5 — Average rating by category
*Which categories have the highest customer satisfaction?*

In [ ]:
rating_cat = (
    df.groupby('product_category')['rating']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'rating': 'avg_rating'})
)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(rating_cat['product_category'], rating_cat['avg_rating'],
              color=sns.color_palette('Blues_d', len(rating_cat)))
ax.set_title('Average Customer Rating by Category')
ax.set_ylabel('Rating (1–5)')
ax.set_ylim(0, 5)
ax.axhline(3, color='gray', linestyle='--', linewidth=1, label='Midpoint (3.0)')
for bar, val in zip(bars, rating_cat['avg_rating']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f'{val:.2f}', ha='center', fontsize=9)
ax.legend()
plt.tight_layout()
plt.savefig('images/05_rating_by_category.png', dpi=150)
plt.show()

print(rating_cat.round(3).to_string(index=False))
print('\nInsight: All categories cluster tightly around 3.0/5.0 — moderate satisfaction. '
      'Books leads (3.02); Beauty trails (2.99). There is improvement room across the board.')

### Query 6. Discount impact on average revenue.  Key business insight
*Do higher discounts hurt profitability?*

In [ ]:
disc_rev = (
    df.groupby('discount_percent')['total_revenue']
    .mean()
    .reset_index()
    .rename(columns={'total_revenue': 'avg_revenue'})
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(disc_rev['discount_percent'], disc_rev['avg_revenue'],
        color='#D85A30', marker='o', linewidth=2.5, markersize=5)
ax.fill_between(disc_rev['discount_percent'], disc_rev['avg_revenue'],
                alpha=0.12, color='#D85A30')
ax.set_title('Avg Revenue per Order vs Discount % — clear inverse relationship')
ax.set_xlabel('Discount (%)')
ax.set_ylabel('Avg Revenue per Order (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.0f}'))

# annotate 0% and 30% points
no_disc = disc_rev[disc_rev['discount_percent'] == 0]['avg_revenue'].values
hi_disc = disc_rev[disc_rev['discount_percent'] == disc_rev['discount_percent'].max()]['avg_revenue'].values
if len(no_disc): ax.annotate(f'No discount\n${no_disc[0]:.0f}',
    xy=(0, no_disc[0]), xytext=(2, no_disc[0]-30), fontsize=9, color='#D85A30')

plt.tight_layout()
plt.savefig('images/06_discount_impact.png', dpi=150)
plt.show()

print(disc_rev.round(2).to_string(index=False))
print('\nInsight: A 20% discount reduces avg revenue by ~18% ($749 → $614). '
      'Discounts do not compensate with higher volume — apply them strategically, not broadly.')

### Query 7. Reviews by category
*Which categories generate the most customer engagement?*

In [ ]:
reviews_cat = (
    df.groupby('product_category')['review_count']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(reviews_cat['product_category'], reviews_cat['review_count'],
       color=sns.color_palette('Blues_d', len(reviews_cat)))
ax.set_title('Total Reviews by Product Category')
ax.set_ylabel('Total Reviews')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.tight_layout()
plt.savefig('images/07_reviews_by_category.png', dpi=150)
plt.show()

print(reviews_cat.to_string(index=False))
print('\nInsight: Beauty and Fashion lead in reviews (>2M each), '
      'reflecting high emotional engagement from buyers in those categories.')

### Query 8. Avg revenue by region & category
*Which region–category combinations yield the highest order value?*

In [ ]:
rev_reg_cat = (
    df.groupby(['product_category', 'customer_region'])['total_revenue']
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={'total_revenue': 'avg_revenue'})
)

pivot = rev_reg_cat.pivot(index='product_category', columns='customer_region', values='avg_revenue')

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(pivot, annot=True, fmt='.0f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Avg Revenue (USD)'})
ax.set_title('Avg Revenue per Order — Category × Region')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('images/08_revenue_region_category.png', dpi=150)
plt.show()

print('\nInsight: Differences across regions for the same category are small (<$50). '
      'Middle East slightly leads for Books; North America for Home & Kitchen.')

### Query 9. Average price by category
*Are products similarly priced across categories?*

In [ ]:
price_cat = (
    df.groupby('product_category')['price']
    .mean()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'price': 'avg_price'})
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(price_cat['product_category'], price_cat['avg_price'],
       color=sns.color_palette('Blues_d', len(price_cat)))
ax.set_title('Average Price by Product Category')
ax.set_ylabel('Avg Price (USD)')
ax.set_ylim(240, 260)
for i, row in price_cat.iterrows():
    ax.text(i, row['avg_price'] + 0.1, f'${row["avg_price"]:.2f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('images/09_avg_price_by_category.png', dpi=150)
plt.show()

print(price_cat.round(2).to_string(index=False))
print('\nInsight: Avg prices span only $1.92 across all categories ($251.89–$253.81). '
      'This suggests standardized mid-tier pricing across the platform.')

### Query 10. Monthly units sold trend
*Does volume follow the same seasonal pattern as revenue?*

In [ ]:
monthly_units = (
    df.groupby('month')['quantity_sold']
    .sum()
    .reset_index()
)
monthly_units['month_str'] = monthly_units['month'].astype(str)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(len(monthly_units)), monthly_units['quantity_sold'],
        color='#185FA5', marker='s', linewidth=2, markersize=4)
ax.set_title('Monthly Units Sold (Jan 2022 – Dec 2023)')
ax.set_xticks(range(len(monthly_units)))
ax.set_xticklabels(monthly_units['month_str'], rotation=45, ha='right', fontsize=8)
ax.axhline(monthly_units['quantity_sold'].mean(), color='gray', linestyle='--',
           linewidth=1, label=f"Avg: {monthly_units['quantity_sold'].mean():,.0f} units")
ax.legend()
plt.tight_layout()
plt.savefig('images/10_monthly_units.png', dpi=150)
plt.show()

print('Insight: Volume mirrors revenue seasonality — Jan spike, Feb dip (~13%), then stable '
      'between 6,100–6,500 units/month. Consistent with post-holiday e-commerce patterns.')

## 5. Correlation analysis

Quantify linear relationships between numeric variables to identify redundancy and key revenue drivers.

In [ ]:
num_cols = ['price', 'discount_percent', 'quantity_sold',
            'rating', 'review_count', 'discounted_price', 'total_revenue']

corr_matrix = df[num_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # show lower triangle only
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Numeric Variables')
plt.tight_layout()
plt.savefig('images/11_correlation_matrix.png', dpi=150)
plt.show()

print('Key correlations:')
print(f"  price ↔ discounted_price : {corr_matrix.loc['price','discounted_price']:.3f}  (expected — derived column)")
print(f"  price ↔ total_revenue    : {corr_matrix.loc['price','total_revenue']:.3f}  (higher price → higher revenue)")
print(f"  qty   ↔ total_revenue    : {corr_matrix.loc['quantity_sold','total_revenue']:.3f}  (volume drives revenue)")
print(f"  disc  ↔ total_revenue    : {corr_matrix.loc['discount_percent','total_revenue']:.3f}  (discounts slightly hurt revenue)")
print(f"  rating ↔ total_revenue   : {corr_matrix.loc['rating','total_revenue']:.3f}  (rating does not drive revenue here)")

## 6. K-Means: Product behavior segments

Group products by price, revenue, and volume to identify strategic product tiers.

In [ ]:
# ---- select features and normalize ----
features_1 = ['price', 'total_revenue', 'quantity_sold']
X1 = df[features_1].copy()

scaler1 = MinMaxScaler()
X1_scaled = scaler1.fit_transform(X1)

# ---- elbow method to confirm k=3 is optimal ----
inertia = []
k_range = range(1, 9)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X1_scaled)
    inertia.append(km.inertia_)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(k_range, inertia, marker='o', color='#185FA5', linewidth=2)
ax.axvline(3, color='#D85A30', linestyle='--', label='k = 3 (chosen)')
ax.set_title('Elbow Method — Optimal Number of Clusters')
ax.set_xlabel('k (number of clusters)')
ax.set_ylabel('Inertia')
ax.legend()
plt.tight_layout()
plt.savefig('images/12_elbow_products.png', dpi=150)
plt.show()

In [ ]:
# ---- fit final model with k=3 ----
km1 = KMeans(n_clusters=3, random_state=42, n_init=10)
df['product_cluster'] = km1.fit_predict(X1_scaled)

# ---- inspect cluster centers ----
centers1 = pd.DataFrame(
    scaler1.inverse_transform(km1.cluster_centers_),
    columns=features_1
).round(2)
centers1.index.name = 'cluster'
print('Cluster centroids (original scale):')
print(centers1)

# ---- assign human-readable labels based on centroids ----
# (adjust mapping if your cluster numbers differ)
segment_map_1 = {
    centers1['total_revenue'].idxmax(): 'High Performance',
    centers1['price'].idxmax(): 'High Price, Low Volume',
    centers1['quantity_sold'].idxmax(): 'Low Price, High Volume'
}
# fallback: label remaining cluster
for i in range(3):
    if i not in segment_map_1:
        segment_map_1[i] = 'Mid Tier'

df['product_segment'] = df['product_cluster'].map(segment_map_1)
print('\nCluster labels assigned:')
print(df['product_segment'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette_1 = {'High Performance': '#185FA5',
             'High Price, Low Volume': '#D85A30',
             'Low Price, High Volume': '#1D9E75',
             'Mid Tier': '#888780'}

# price vs revenue
for seg, grp in df.groupby('product_segment'):
    axes[0].scatter(grp['price'], grp['total_revenue'],
                    alpha=0.3, s=8, label=seg, color=palette_1.get(seg, 'gray'))
axes[0].set_title('Price vs Total Revenue')
axes[0].set_xlabel('Price (USD)')
axes[0].set_ylabel('Total Revenue (USD)')
axes[0].legend(fontsize=8, markerscale=2)

# price vs quantity
for seg, grp in df.groupby('product_segment'):
    axes[1].scatter(grp['price'], grp['quantity_sold'],
                    alpha=0.3, s=8, label=seg, color=palette_1.get(seg, 'gray'))
axes[1].set_title('Price vs Units Sold')
axes[1].set_xlabel('Price (USD)')
axes[1].set_ylabel('Quantity Sold')

plt.suptitle('K-Means Product Segments (k=3)', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('images/13_kmeans_products.png', dpi=150, bbox_inches='tight')
plt.show()

print(df.groupby('product_segment')[features_1].mean().round(2))

## 7. K-Means: Customer satisfaction segments

Group customers by rating, review activity, and discount sensitivity.

In [ ]:
features_2 = ['rating', 'review_count', 'discount_percent']
X2 = df[features_2].copy()

scaler2 = MinMaxScaler()
X2_scaled = scaler2.fit_transform(X2)

km2 = KMeans(n_clusters=3, random_state=42, n_init=10)
df['satisfaction_cluster'] = km2.fit_predict(X2_scaled)

centers2 = pd.DataFrame(
    scaler2.inverse_transform(km2.cluster_centers_),
    columns=features_2
).round(3)

# label by rating level
sat_map = {
    centers2['rating'].idxmax(): 'Highly Satisfied',
    centers2['rating'].idxmin(): 'Low Satisfaction',
    centers2['discount_percent'].idxmax(): 'Discount Driven'
}
for i in range(3):
    if i not in sat_map:
        sat_map[i] = 'Moderate'

df['satisfaction_segment'] = df['satisfaction_cluster'].map(sat_map)

print('Satisfaction cluster centroids:')
print(centers2)
print('\nSegment distribution:')
print(df['satisfaction_segment'].value_counts())

fig, ax = plt.subplots(figsize=(9, 4))
palette_2 = {'Highly Satisfied': '#1D9E75', 'Low Satisfaction': '#D85A30',
             'Discount Driven': '#185FA5', 'Moderate': '#888780'}
for seg, grp in df.groupby('satisfaction_segment'):
    ax.scatter(grp['discount_percent'], grp['rating'],
               alpha=0.3, s=8, label=seg, color=palette_2.get(seg, 'gray'))
ax.set_title('Customer Satisfaction Segments')
ax.set_xlabel('Discount (%)')
ax.set_ylabel('Rating')
ax.legend(fontsize=8, markerscale=2)
plt.tight_layout()
plt.savefig('images/14_kmeans_satisfaction.png', dpi=150)
plt.show()

## 8. Key findings & recommendations

Summary of actionable insights derived from the analysis.

In [ ]:
findings = {
    'Revenue is balanced across categories': (
        'All 6 categories generate ~16.5% of total revenue each. '
        'No single category creates dependency risk.'
    ),
    'Discounts above 20% hurt profitability': (
        'Avg order revenue drops 18% from $749 (0%) to $614 (20%) '
        'without compensating volume gains. Recommend capping discounts at 15%.'
    ),
    'January post-holiday spike is consistent': (
        'Jan is the top revenue month in both 2022 and 2023, followed by a ~13% '
        'Feb dip. Inventory and marketing should anticipate this pattern.'
    ),
    'Payment adoption is equally distributed': (
        'All 5 methods account for ~20% each. Wallet and UPI lead marginally, '
        'signaling strong digital payment maturity.'
    ),
    '3 distinct product segments identified': (
        'High Performance (high price + high volume), '
        'High Price Low Volume (expensive, low demand), '
        'Low Price High Volume (affordable, mass market).'
    ),
    'Discount-driven customers have lower ratings': (
        'Customers who buy primarily under heavy discounts rate products lower '
        'than satisfied buyers — discounts may attract the wrong audience.'
    ),
}

print('=== KEY FINDINGS & RECOMMENDATIONS ===')
for i, (title, detail) in enumerate(findings.items(), 1):
    print(f'\n{i}. {title}')
    print(f'   {detail}')

---

**Tools used:** Python · Pandas · Matplotlib · Seaborn · Scikit-learn  
**Original analysis tool:** KNIME Analytics Platform  
**Dataset:** [Amazon Sales Dataset on Kaggle](https://www.kaggle.com/datasets/aliiihussain/amazon-sales-dataset)